# Mini-Batch Gradient Descent

Gradient Descent optimization comes in three main variants depending on how much data is processed before updating the model parameters. **Mini-Batch Gradient Descent** serves as the optimal middle ground between the stability of Batch Gradient Descent (BGD) and the extreme speed of Stochastic Gradient Descent (SGD).

---

## Summary of Gradient Descent Variants

| Variant | Data per Update | Updates per Epoch | Pros | Cons |
| :--- | :--- | :--- | :--- | :--- |
| **Batch Gradient Descent (BGD)** | Entire Dataset ($n$ rows) | $1$ | Smooth, stable, guaranteed convergence | Extremely slow on big data; RAM limitations |
| **Stochastic Gradient Descent (SGD)** | Single Row ($1$ sample) | $n$ | Extremely fast; fits in RAM; escapes local minima | Noisy, zigzagging path; never settles without decay |
| **Mini-Batch Gradient Descent (MBGD)** | Predefined Batch Size $B$ (e.g., $32, 64, 128$) | $n / B$ | **Balances stability and speed; leverages vectorized hardware speedups** | Introduces batch size $B$ as a new hyperparameter |

---


## 1. Comparing the Gradient Descent Variants

### 1.1 Batch Gradient Descent (BGD)
- BGD processes the **entire dataset** to calculate the gradients before updating the parameters in a single step.
- If $n = 100,000$, BGD runs 1 update per epoch, calculating $100,000$ gradients. While stable, this is computationally prohibitive and requires all data to fit in RAM.

### 1.2 Stochastic Gradient Descent (SGD)
- SGD updates the parameters after seeing **each individual training sample**.
- If $n = 100,000$, SGD performs $100,000$ updates per epoch. It is fast and highly memory-efficient, but its path is extremely volatile and chaotic.

### 1.3 Mini-Batch Gradient Descent
- Mini-Batch GD splits the dataset into small, manageable subsets of size $B$ (e.g., $B = 32, 64$ or $128$).
- If $n = 1,000$ and $B = 100$, the algorithm updates parameters $10$ times per epoch ($1,000 / 100 = 10$ batches).
- **The Sweet Spot:** MBGD averages out the noise across the $B$ samples, resulting in a much smoother convergence path than SGD. At the same time, it is far faster than BGD because it updates weights multiple times per epoch and easily fits within RAM limits.

## 2. Mathematical Formulation of Mini-Batch GD

Let $S$ be a mini-batch subset containing $B$ indices randomly sampled from the dataset $\{1, 2, \dots, n\}$.

Let $X_S$ be the input feature matrix for the mini-batch of size $B \times (m+1)$ (with the intercept column of 1s prepended), and $Y_S$ be the target vector of size $B \times 1$.

The predicted target vector $\hat{Y}_S$ for the mini-batch is:

$$\hat{Y}_S = X_S \beta$$

We calculate the gradient vector $\nabla L_S(\beta)$ over the mini-batch by averaging the derivatives of the $B$ samples:

$$\nabla L_S(\beta) = -\frac{2}{B} X_S^T (Y_S - X_S\beta)$$

### Vectorized Update Rule:
We update the entire parameter vector $\beta$ simultaneously using the mini-batch gradient:

$$\beta_{new} = \beta_{old} - \eta \cdot \nabla L_S(\beta)$$

$$\beta_{new} = \beta_{old} + \frac{2\eta}{B} X_S^T (Y_S - X_S\beta)$$

Where $\eta$ is the learning rate. This formulation leverages **vectorized matrix multiplication**, allowing modern multi-core CPUs and GPUs to perform updates extremely fast.

## 3. Code Setup & Data Generation

We will import the required scientific libraries, generate a synthetic multi-feature regression dataset, and establish a baseline OLS Linear Regression model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Set display style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Generate synthetic regression dataset
np.random.seed(42)
X, y = make_regression(n_samples=200, n_features=3, noise=12, random_state=42)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape:  {X_test.shape}")


In [ ]:
# Fit standard OLS Linear Regression as baseline
ols = LinearRegression()
ols.fit(X_train, y_train)

print("OLS BASELINE RESULTS:")
print(f"Intercept:     {ols.intercept_:.6f}")
print(f"Coefficients:  {ols.coef_}")
print(f"Test R2 Score:  {r2_score(y_test, ols.predict(X_test)):.6f}")


## 4. Implementation from Scratch

Let's build a custom `MiniBatchGDRegressor` class from scratch using NumPy. To perform mini-batch updates:

1. **Define Batch Size:** Introduce a `batch_size` parameter in the initializer.
2. **Loop Structure:** Split the training loop so it iterates through batches instead of single rows or the full dataset.
3. **Random Selection:** Shuffle the training indices at the start of each epoch to partition random mini-batches without replacement (standard mini-batch execution).
4. **Update Weights:** Extract the indices for each batch, index the dataset using fancy indexing (`X_train[batch_idx]`), calculate the batch derivative, and update parameters.

In [ ]:
class MiniBatchGDRegressor:
    """Custom Mini-Batch Gradient Descent Regressor"""
    def __init__(self, batch_size=32, learning_rate=0.01, epochs=100):
        self.batch_size = batch_size
        self.lr = learning_rate
        self.epochs = epochs
        self.coef_ = None
        self.intercept_ = None
        self.history = []  # Logs parameters and overall MSE at each epoch
        
    def fit(self, X, y):
        # Add column of 1s at index 0 to represent intercept
        X_new = np.insert(X, 0, 1, axis=1)
        n_samples, n_features = X_new.shape
        beta = np.zeros(n_features)  # Initialize beta vector to zeros
        
        for epoch in range(self.epochs):
            # 1. Shuffle indices at the start of each epoch
            indices = np.arange(n_samples)
            np.random.shuffle(indices)
            X_shuffled = X_new[indices]
            y_shuffled = y[indices]
            
            # 2. Iterate through data in strides of batch_size
            for i in range(0, n_samples, self.batch_size):
                # Extract batch data
                X_batch = X_shuffled[i : i + self.batch_size]
                y_batch = y_shuffled[i : i + self.batch_size]
                
                # Calculate predictions for current batch
                y_pred_batch = np.dot(X_batch, beta)
                
                # Calculate gradients over the batch size
                # d_beta = -2/B * X_batch^T * (y_batch - y_pred_batch)
                d_beta = - (2 / len(X_batch)) * np.dot(X_batch.T, y_batch - y_pred_batch)
                
                # Update parameter vector
                beta = beta - self.lr * d_beta
                
            # Calculate overall dataset MSE at the end of the epoch
            y_pred_epoch = np.dot(X_new, beta)
            loss_epoch = np.mean((y - y_pred_epoch)**2)
            self.history.append((beta.copy(), loss_epoch))
            
        self.intercept_ = beta[0]
        self.coef_ = beta[1:]
        
    def predict(self, X):
        return np.dot(X, self.coef_) + self.intercept_


In [ ]:
# Fit custom Mini-Batch Regressor
mbgd = MiniBatchGDRegressor(batch_size=20, learning_rate=0.05, epochs=250)
mbgd.fit(X_train, y_train)

y_pred_mbgd = mbgd.predict(X_test)
r2_mbgd = r2_score(y_test, y_pred_mbgd)

print("="*60)
print("MINI-BATCH GD REGRESSOR RESULTS")
print("="*60)
print(f"Intercept:      {mbgd.intercept_:.6f} (OLS: {ols.intercept_:.6f})")
print(f"Coefficients:   {mbgd.coef_}")
print(f"OLS Coefficients: {ols.coef_}")
print(f"Test R2 Score:   {r2_mbgd:.6f} (OLS: {r2_score(y_test, ols.predict(X_test)):.6f})")
print("="*60)


## 5. Visualizing Path to Convergence: BGD vs. SGD vs. Mini-Batch GD

To gain physical intuition, we will generate a 1D dataset and trace the optimization path of the three algorithms on a 2D contour plot of the MSE Loss Surface.

- **Batch GD (Green):** Direct, perpendicular path with zero noise. Extremely smooth, but slow.
- **Stochastic GD (Red):** Extremely volatile, chaotic path. Jumps around in random directions due to single-sample updates.
- **Mini-Batch GD (Blue):** A smooth, balanced trajectory. It features small fluctuations, but moves steadily and directly toward the global minimum, combining the speed of SGD with the stability of BGD.

In [ ]:
# Implement simple BGD and SGD regressors for visual tracking
class BGDRegressorVisual:
    def __init__(self, lr=0.1, epochs=30):
        self.lr = lr
        self.epochs = epochs
        self.history = []
    def fit(self, X, y):
        X_new = np.insert(X, 0, 1, axis=1)
        n_samples, n_features = X_new.shape
        beta = np.zeros(n_features)
        for epoch in range(self.epochs):
            y_pred = np.dot(X_new, beta)
            loss = np.mean((y - y_pred)**2)
            self.history.append((beta.copy(), loss))
            d_beta = - (2 / n_samples) * np.dot(X_new.T, y - y_pred)
            beta = beta - self.lr * d_beta

class SGDRegressorVisual:
    def __init__(self, lr=0.02, epochs=30):
        self.lr = lr
        self.epochs = epochs
        self.history = []
    def fit(self, X, y):
        X_new = np.insert(X, 0, 1, axis=1)
        n_samples, n_features = X_new.shape
        beta = np.zeros(n_features)
        for epoch in range(self.epochs):
            indices = np.arange(n_samples)
            np.random.shuffle(indices)
            for idx in indices:
                xi = X_new[idx]
                yi = y[idx]
                y_pred = np.dot(xi, beta)
                d_beta = -2 * (yi - y_pred) * xi
                beta = beta - self.lr * d_beta
            # Calculate overall loss at end of epoch
            y_pred_epoch = np.dot(X_new, beta)
            loss_epoch = np.mean((y - y_pred_epoch)**2)
            self.history.append((beta.copy(), loss_epoch))


In [ ]:
# Generate simple 1D dataset
np.random.seed(42)
X_simple = np.random.uniform(-2, 2, 100).reshape(-1, 1)
y_simple = 2.5 * X_simple.ravel() + 1.2 + np.random.normal(0, 0.4, 100)

# Train models on simple dataset
bgd = BGDRegressorVisual(lr=0.08, epochs=40)
bgd.fit(X_simple, y_simple)

sgd = SGDRegressorVisual(lr=0.02, epochs=40)
sgd.fit(X_simple, y_simple)

mbgd_vis = MiniBatchGDRegressor(batch_size=15, learning_rate=0.05, epochs=40)
mbgd_vis.fit(X_simple, y_simple)

# Create contour grid for slope m and intercept b
m_vals = np.linspace(-1, 5, 100)
b_vals = np.linspace(-1, 3, 100)
M, B = np.meshgrid(m_vals, b_vals)
Z = np.zeros(M.shape)

for i in range(len(m_vals)):
    for j in range(len(b_vals)):
        pred = M[j, i] * X_simple.ravel() + B[j, i]
        Z[j, i] = np.mean((y_simple - pred)**2)

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

# Plot 1: Contour Plot with Optimization Paths
contours = axes[0].contour(M, B, Z, levels=35, cmap='viridis')
axes[0].clabel(contours, inline=True, fontsize=8)

bgd_betas = np.array([h[0] for h in bgd.history])
sgd_betas = np.array([h[0] for h in sgd.history])
mbgd_betas = np.array([h[0] for h in mbgd_vis.history])

axes[0].plot(bgd_betas[:, 1], bgd_betas[:, 0], 'o-', color='#2ecc71', linewidth=2.5, markersize=5, label='BGD (Smooth & Rigid)')
axes[0].plot(sgd_betas[:, 1], sgd_betas[:, 0], 'x-', color='#e74c3c', linewidth=1.5, markersize=5, alpha=0.6, label='SGD (Highly Volatile)')
axes[0].plot(mbgd_betas[:, 1], mbgd_betas[:, 0], 's-', color='#3498db', linewidth=2.0, markersize=5, alpha=0.9, label='Mini-Batch GD (Balanced)')
axes[0].scatter([2.5], [1.2], color='black', marker='*', s=200, zorder=10, label='Global Minimum')

axes[0].set_title('Optimization Paths on Loss Surface Contour', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Slope (m)')
axes[0].set_ylabel('Intercept (b)')
axes[0].legend(loc='lower left')

# Plot 2: Epoch vs Loss Curve
axes[1].plot([h[1] for h in bgd.history], color='#2ecc71', linewidth=2.5, label='BGD Loss')
axes[1].plot([h[1] for h in sgd.history], color='#e74c3c', linewidth=2.0, alpha=0.6, label='SGD Loss')
axes[1].plot([h[1] for h in mbgd_vis.history], color='#3498db', linewidth=2.0, alpha=0.9, label='Mini-Batch GD Loss')

axes[1].set_title('MSE Loss Convergence Curves', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Mean Squared Error')
axes[1].legend()

plt.tight_layout()
plt.show()


## 6. Scikit-Learn Implementation: The `partial_fit` Workaround

In Scikit-Learn, the standard `SGDRegressor` does not feature a `batch_size` parameter because it is built strictly as a stochastic gradient solver. However, Scikit-Learn offers a powerful alternative: the **`partial_fit`** method.

`partial_fit` executes a single update step on the model coefficients using only the subset of data passed to it. By wrapping `partial_fit` in a custom training loop that manually breaks the dataset into mini-batches, we can implement custom Mini-Batch Gradient Descent using Scikit-Learn's optimized solvers.

> **Note:** When using `partial_fit` for the first time, we must scale features correctly using `StandardScaler` (scaling fits to the training set and transforms both sets).

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler

# 1. Feature scaling (Crucial for gradient descent)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Initialize SGDRegressor model
sgd_reg = SGDRegressor(learning_rate='constant', eta0=0.01, random_state=42)

# 3. Custom loop to simulate Mini-Batch GD using partial_fit
batch_size = 32
epochs = 200
n_samples = X_train_scaled.shape[0]

for epoch in range(epochs):
    # Shuffle data indices at each epoch
    indices = np.arange(n_samples)
    np.random.shuffle(indices)
    X_shuffled = X_train_scaled[indices]
    y_shuffled = y_train[indices]
    
    # Process data in batches and update coefficients
    for i in range(0, n_samples, batch_size):
        X_batch = X_shuffled[i : i + batch_size]
        y_batch = y_shuffled[i : i + batch_size]
        
        # Fit incrementally on batch
        sgd_reg.partial_fit(X_batch, y_batch)

# 4. Predict and evaluate
y_pred_scaled = sgd_reg.predict(X_test_scaled)
r2_sgd_scaled = r2_score(y_test, y_pred_scaled)

print("="*60)
print("SCIKIT-LEARN PARTIAL_FIT MINI-BATCH RESULTS")
print("="*60)
print(f"Intercept:      {sgd_reg.intercept_[0]:.6f}")
print(f"Coefficients:   {sgd_reg.coef_}")
print(f"Test R2 Score:   {r2_sgd_scaled:.6f}")
print("="*60)


## Summary Checklist for Mini-Batch Gradient Descent

1. **Choose an Appropriate Batch Size ($B$):** Standard batch sizes are powers of 2 (e.g., $32, 64, 128, 256$). Smaller batches behave more like SGD (noisier, escape local minima), while larger batches behave like BGD (smoother, faster matrix operations).
2. **Vectorization Speedups:** Batch sizes are chosen to fit the caching structure of modern hardware (CPUs/GPUs). Mini-Batch GD utilizes SIMD matrix architecture to run computations faster than single-row loops.
3. **Shuffle the Data:** Just like SGD, ensure training data indices are randomized at each epoch so that the mini-batches do not represent sequential biases.
4. **Manually Iterate using `partial_fit`:** If implementing in Scikit-Learn, use the `partial_fit` custom batching workaround to train models incrementally on streaming or large out-of-core datasets.